In [ ]:
# ruff: noqa: E402
from __future__ import annotations

import os
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

while Path.cwd().name in ["notebooks", "evaluation"]:
    os.chdir("..")

from config.config import OUTPUTS_DIR

plt.rcParams["figure.dpi"] = 144

## Error rates


In [ ]:
datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}

models = [
    "gemini-2.0-flash-lite",
]

experiment_type = "n_few_shots"

experiments = [
    f"v{version}/{n}" for version in range(5) for n in [0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75]
]
experiments += [f"v{version}/{n}" for version in range(5, 10) for n in [1, 2, 3, 5, 8, 11]]

suffix = "_reduced"
records = []

for dataset, subsets in datasets_map.items():
    for subset in subsets:
        for model in models:
            for exp in experiments:
                run_subdir = f"{dataset}/{subset}/{model}/{experiment_type}/{exp}"
                path = OUTPUTS_DIR / run_subdir / f"results{suffix}.csv"

                if not path.exists() or path.stat().st_size < 2:
                    continue

                df_local = pd.read_csv(path)

                if "Prompt" not in df_local.columns:
                    continue

                for prompt in df_local["Prompt"].unique():
                    records.append(
                        {
                            "Dataset": dataset,
                            "Subset": subset,
                            "Model": model,
                            "Experiment": exp,
                            "Prompt": prompt,
                            "Exists": True,
                        }
                    )

df = pd.DataFrame(records)
df["TaskPrompt"] = df["Dataset"] + "/" + df["Subset"] + " | " + df["Prompt"]

for model in df["Model"].unique():
    model_df = df[df["Model"] == model]

    pivot = model_df.pivot_table(
        index="TaskPrompt", columns="Experiment", values="Exists", aggfunc="first", fill_value=False
    )
    # print missing experiments in format task, prompt, n_few_shots_version/number from pivot table
    missing = pivot[pivot == False].stack().reset_index()
    for _, row in missing.iterrows():
        print(f"{row['TaskPrompt']}, {row['Experiment']}")

    plt.figure(figsize=(16, max(6, 0.3 * len(pivot))))
    sns.heatmap(
        pivot,
        cmap=["#ffcccc", "#c6efce"],  # red = missing, green = exists
        cbar=False,
        linewidths=0.4,
        linecolor="gray",
    )

    plt.title(f"Missing Experiments for {model} (Grouped by Task & Prompt)", fontsize=14)
    plt.xlabel("Few-shot version / number")
    plt.ylabel("Task | Prompt")
    plt.xticks(rotation=90, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}

models = [
    "gemini-2.0-flash-lite",
    # "mistral.mistral-large-2402-v1:0",
    # "gemini-2.5-flash-lite",
]

experiment_type = "n_few_shots"
few_shot_versions = [f"v{v}" for v in range(10)]
few_shot_numbers = [0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75]
suffix = "_reduced"
records = []

for dataset, subsets in datasets_map.items():
    for subset in subsets:
        for model in models:
            for v, n in product(few_shot_versions, few_shot_numbers):
                run_dir = OUTPUTS_DIR / dataset / subset / model / experiment_type / f"{v}/{n}"
                if not run_dir.exists():
                    continue

                for csv_file in run_dir.glob(f"*_results{suffix}.csv"):
                    prompt_name = csv_file.stem.replace(f"_results{suffix}", "")
                    if f"{model}_" in prompt_name:
                        prompt_name = prompt_name.split(f"{model}_")[-1]

                    try:
                        df = pd.read_csv(csv_file)
                    except Exception as e:
                        print(f"Error reading {csv_file}: {e}")
                        continue

                    if "Prediction" not in df.columns:
                        print(f"Missing 'Prediction' column in {csv_file}")
                        continue

                    total = len(df)
                    if total == 0:
                        print(f"Empty DataFrame in {csv_file}")
                        continue

                    false_count = (df["Prediction"].astype(str).str.lower() == "error").sum()
                    error_rate = false_count / total

                    if error_rate == 0:
                        continue

                    records.append(
                        {
                            "Task": f"{dataset}/{subset}",
                            "Model": model,
                            "Prompt": prompt_name,
                            "FewShot": f"{v}/{n}",
                            "ErrorRate": error_rate,
                        }
                    )

results_df = pd.DataFrame(records)
if results_df.empty:
    print("No error records found.")
else:
    print("Total error records found:", len(results_df))

    results_df["Label"] = results_df["Task"] + "/" + results_df["Prompt"] + "/" + results_df["FewShot"]

    plt.figure(figsize=(max(14, len(results_df) * 0.25), 7))
    sns.barplot(
        data=results_df.sort_values("ErrorRate", ascending=False),
        x="Label",
        y="ErrorRate",
        hue="Model",
        palette="Reds",
    )

    plt.title("Error Rate per Task / Prompt / Few-Shot", fontsize=15)
    plt.ylabel("Error Rate (Error / Total)")
    plt.xlabel("Task / Prompt / Few-Shot")
    plt.xticks(rotation=90, ha="center", fontsize=8)
    plt.legend(title="Model")
    plt.tight_layout()
    # plt.yscale("log")
    plt.grid(axis="y", alpha=0.3)

In [ ]:
datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}

models = [
    # "mistral.mistral-large-2402-v1:0",
    "gemini-2.0-flash-lite",
]

experiment_type = "n_few_shots"
few_shot_versions = ["v0", "v1", "v2", "v3", "v4"]
few_shot_numbers = [0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75]
suffix = "_reduced"

records = []

for dataset, subsets in datasets_map.items():
    for subset in subsets:
        for model in models:
            for n, v in product(few_shot_numbers, few_shot_versions):
                run_dir = OUTPUTS_DIR / dataset / subset / model / experiment_type / f"{v}/{n}"
                if not run_dir.exists():
                    continue

                for csv_file in run_dir.glob(f"*_results{suffix}.csv"):
                    prompt_name = csv_file.stem.replace(f"_results{suffix}", "")
                    prompt_name = prompt_name.split(f"{model}_")[-1]

                    try:
                        df = pd.read_csv(csv_file)
                    except Exception as e:
                        print(f"Error reading {csv_file}: {e}")
                        continue

                    if "Prediction" not in df.columns:
                        print(f"Missing 'Prediction' column in {csv_file}")
                        continue

                    counts = df["Prediction"].value_counts().to_dict()
                    total = len(df)

                    for pred_label, count in counts.items():
                        records.append(
                            {
                                "Dataset": dataset,
                                "Subset": subset,
                                "Task": f"{dataset}/{subset}",
                                "Model": model,
                                "FewShot": f"{v}/{n}",
                                "Prompt": prompt_name,
                                "Prediction": str(pred_label),
                                "Count": count,
                                "Total": total,
                            }
                        )

results_df = pd.DataFrame(records)

results_df["Proportion"] = results_df["Count"] / results_df["Total"]

sns.set_theme(style="whitegrid")

for (task, prompt), group_df in results_df.groupby(["Task", "Prompt"]):
    plt.figure(figsize=(14, 4))
    # group_df = group_df[group_df["Prediction"].isin(["error", "no_match", "match"])]
    sns.barplot(
        data=group_df,
        x="FewShot",
        y="Proportion",
        hue="Prediction",
        palette="Set2",
    )
    plt.title(f"{task} — {prompt}")
    plt.ylabel("Proportion of Predictions")
    plt.xlabel("Few-shot version / number")
    plt.xticks(rotation=90, ha="right")
    plt.legend(title="Prediction")
    plt.tight_layout()
    plt.show()
